In [ ]:
import pandas as pd
import commons as c
import plotly.express as px
import plotly.graph_objects as go
import scipy.stats as stats
import scikit_posthocs as sp
import numpy as np


# Get datasets

In [ ]:
csv_path = 'results/dataframes/results_all_mutants.csv'
df = pd.read_csv(csv_path, dtype=c.type_dict)

# Get Box Plots

In [ ]:
category_names = {
    'Qubits_number': 'Number of qubits', 
    'gates': 'Number of gates', 
    'depth': 'Circuit depth',
    'Algorithm': 'Algorithm name', 
    'Input_type': 'Type of input', 
    'Output_type': 'Type of output',
    'Gate_type': 'Mutated gate', 
    'Operator': 'Mutation operator', 
    'Relative_position': 'Relative position of the mutation'
}

In [ ]:
def print_box_plot(df, cat, distance, output_folder, file_name, median_line=False, width=2000):
    df = df.copy()  # Ensure it's a copy
    
    cat_range = sorted(df[cat].unique())  # Extract unique values from the column   
    
    # Create the box plot
    fig = px.box(
        df, 
        y=distance, 
        x=cat, 
        color="nature", 
        category_orders={
            cat: cat_range,
            "nature": ["Equivalent mutant", "Non-Equivalent mutant"]
        },
        # title="Boxplot of Distance by noise model and program type",
        labels={cat: category_names[cat], distance: "Distance", 'nature': "Legend"},
        points=False,
        boxmode="group"
    ) 
    
    # Compute medians for each category and true_label
    median_values = df.groupby([cat, "nature"])[distance].median().reset_index()
    
    if median_line:
        # Define colors matching the boxplot
        colors = {"Equivalent mutant": "blue", "Non-Equivalent mutant": "red"}
        
        for nature in median_values["nature"].unique():
            subset = median_values[median_values["nature"] == nature]
            fig.add_trace(go.Scatter(
                x=subset[cat], 
                y=subset[distance], 
                mode='lines',
                name=f"Median - {nature}",
                line=dict(color=colors[nature]) #, dash='dot')
            ))
    
    # Adjust layout for better visualization
    fig.update_layout(
        xaxis=dict(tickmode="array", tickvals=cat_range)
    )
    
    # Save the figure
    c.setup_layout_and_save(fig, output_folder, file_name, yaxis_range=[0, 1], width=width)

In [ ]:
def category_plot(df, m):
    category_groups = [
        ('Qubits_number', "RQ2_1", True, 3000), # 7
        ('gates', "RQ2_1", True, 3000), # 65
        ('depth', "RQ2_1", True, 3000), # 42
        ('Algorithm', "RQ2_2", False, 2000), # 5
        ('Input_type', "RQ2_2", False, 1000), # 2
        ('Output_type', "RQ2_2", False, 1000), # 2
        ('Gate_type', "RQ2_3", False, 1000), # 2
        ('Operator', "RQ2_3", False, 1000), # 3
        ('Relative_position', "RQ2_3", True, 2000) # 5
    ]
    
    for hw in c.hardware:
        df_hw = df[df['hardware'] == hw]
        df_metric = df_hw[df_hw['metric'] == m]

        for cat, subfolder, showmedian, width in category_groups:
            selected_columns = df_metric[[cat, 'nature', 'ideal_distance', 'noisy_distance']]
            file_name = f'{hw}_{cat}'
                
            for distance_type in ['noisy_distance', 'ideal_distance']:
                variant = 'noisy' if distance_type == 'noisy_distance' else 'ideal'
                output_folder = f'results/RQ2/{subfolder}/{variant}/{m}'
                print_box_plot(selected_columns, cat, distance_type, output_folder, file_name, showmedian, width)


In [ ]:
m = "T"
category_plot(df, m)

m = "H"
category_plot(df, m)

# Statistical tests

In [ ]:
df

In [ ]:
# Encode the relative position as numbers to analyze correlation
label_map = {
    'Beginning':0,
    'Pre middle': 1,
    'Middle': 2,
    'Post middle': 3,
    'End': 4
}

# Create a new column with the mapped integer values
df['Relative_position_encoding'] = df['Relative_position'].map(label_map)
df

In [ ]:

correlation_results = []

# Loop over each metric
for nature in ['Equivalent mutant', 'Non-Equivalent mutant']:
    df_nature = df[df['nature'] == nature]
    for metric in df_nature['metric'].unique():
        subset_df = df_nature[df_nature['metric'] == metric]
    
        for target in ['ideal_distance', 'noisy_distance']:
            for var in ['Qubits_number', 'depth', 'gates', 'Relative_position_encoding']:
                x = subset_df[var].dropna()
                y = subset_df[target].dropna()
                
                # Ensure matching indices after dropna
                common_idx = x.index.intersection(y.index)
                x, y = x.loc[common_idx], y.loc[common_idx]
                
                if len(x) < 2:
                    r, p = np.nan, np.nan
                else:
                    r, p = stats.pearsonr(x, y)
                
                abs_r = abs(r) if pd.notna(r) else np.nan
                if pd.isna(abs_r):
                    strength = "N/A"
                elif abs_r < 0.10:
                    strength = "Negligible"
                elif abs_r < 0.30:
                    strength = "Weak"
                elif abs_r < 0.50:
                    strength = "Moderate"
                else:
                    strength = "Strong"
                
                correlation_results.append({
                    'Metric': metric,
                    'Nature': nature,
                    'Variable': var,
                    'Target': target,
                    'Correlation': r,
                    'p-value': p,
                    'Strength': strength
                })

# Create DataFrame
correlation_df = pd.DataFrame(correlation_results)
correlation_df


In [ ]:

categorical_results = []

categorical_vars = ['Algorithm','Output_type','Input_type', 'Gate_type','Operator'] # Replace with yours

for nature in ['Equivalent mutant', 'Non-Equivalent mutant']:
    df_nature = df[df['nature'] == nature]

    for metric in df_nature['metric'].unique():
        subset_df = df_nature[df_nature['metric'] == metric]

        for target in ['ideal_distance', 'noisy_distance']:
            for cat_var in categorical_vars:
                pair_relations = []
                pair_relations_not = []
                if cat_var not in subset_df.columns:
                    continue

                group_data = subset_df[[cat_var, target]].dropna()
                unique_groups = group_data[cat_var].unique()

                if len(unique_groups) < 2:
                    continue

                grouped_values = [group_data[group_data[cat_var] == g][target].values for g in unique_groups]
                n_total = sum(len(g) for g in grouped_values)

                if len(unique_groups) == 2:
                    # Mann–Whitney U Test
                    stat, p = stats.mannwhitneyu(grouped_values[0], grouped_values[1], alternative='two-sided')
                    test_used = "Mann–Whitney U"
                    if p < 0.05:
                        pair_relations.append(f'{unique_groups[0]} - {unique_groups[1]}: p-val: {p}')
                    else:
                        pair_relations_not.append(f'{unique_groups[0]} - {unique_groups[1]}: p-val: {p}')

                else:
                    # Kruskal–Wallis Test
                    stat, p = stats.kruskal(*grouped_values)
                    test_used = "Kruskal–Wallis"
                    p_values = sp.posthoc_dunn(subset_df, target, cat_var, p_adjust='holm')
                    
                    # Get boolean mask of significant results
                    significant_mask = (p_values < 0.05)
                    
                    # Loop through the upper triangle of the matrix (to avoid duplicates)
                    for i in range(len(p_values)):
                        for j in range(i + 1, len(p_values)):
                            if significant_mask.iloc[i, j]:
                                group1 = p_values.index[i]
                                group2 = p_values.columns[j]
                                p_val = p_values.iloc[i, j]
                                pair_relations.append(f'{group1} - {group2}: p-val: {p_val}')
                            else:
                                group1 = p_values.index[i]
                                group2 = p_values.columns[j]
                                p_val = p_values.iloc[i, j]
                                pair_relations_not.append(f'{group1} - {group2}: p-val: {p_val}')

                categorical_results.append({
                    'Metric': metric,
                    'Nature': nature,
                    'Categorical Variable': cat_var,
                    'Target': target,
                    'Test': test_used,
                    'Stat': stat,
                    'p-value': p,
                    'significant_pairs': pair_relations,
                    'not-significant_pairs': pair_relations_not

                    # 'Effect Size (r)': r,
                    # 'Strength': strength
                })
                # pair_relations.clear()

# Create and print result DataFrame
categorical_df = pd.DataFrame(categorical_results)
categorical_df
